# Chapter 11 — Knowing When Not to Answer

**Book alignment:** Hallucination From First Principles, Chapter 11

**Question this notebook isolates:** Does a selective gate trade coverage against risk along a curve where always-abstain is perfect-risk zero-value?

All evidence states below are deterministic synthetic fixtures; they demonstrate the mechanism type, not real model behavior.

In [ ]:
import random
import numpy as np

random.seed(11)
rng = np.random.default_rng(11)

print("seed fixed:", 11)

## Answerability contract: coverage hides a critical gap

Four of five required fields are SUPPORTED, yet the one missing field is the answer itself (Q3 revenue). Weighted coverage looks healthy; the typed record still routes to retrieval, and only the supported subclaim survives at claim granularity.

In [ ]:
contract = {
    "question_type": "reported_financial_value",
    "target": "company_q3_revenue",
    "required_fields": ["company_identity", "reporting_period", "revenue_value", "currency"],
    "critical_fields": ["reporting_period", "revenue_value"],
}

evidence = {"company_identity": "SUPPORTED", "reporting_period": "SUPPORTED",
            "revenue_value": "INSUFFICIENT", "currency": "SUPPORTED"}

coverage = sum(v == "SUPPORTED" for v in evidence.values()) / len(evidence)
critical_missing = [f for f in contract["critical_fields"] if evidence[f] != "SUPPORTED"]
state = "INSUFFICIENT_EVIDENCE" if critical_missing else "ANSWERABLE"
next_action = "RETRIEVE" if critical_missing else "ANSWER"

# claim-level partial answer: Q2 supported, Q3 withheld
claims = [("Q2 revenue was $43.8 million.", "SUPPORTED", "ACCEPT"),
          ("Q3 revenue was ~$46 million.", "INSUFFICIENT", "OMIT_ABSTAIN_WITH_GAP")]

print(f"coverage={coverage:.2f} critical_missing={critical_missing} state={state} next={next_action}")
for text, st, gate in claims:
    print(f"  [{gate:20s}] ({st:12s}) {text}")

In [ ]:
assert abs(coverage - 0.75) < 1e-9
assert critical_missing == ["revenue_value"]
assert state == "INSUFFICIENT_EVIDENCE" and next_action == "RETRIEVE"
assert claims[0][2] == "ACCEPT" and claims[1][2] == "OMIT_ABSTAIN_WITH_GAP"
print("PASS: high coverage with a critical gap is unanswerable, yet partially salvageable")

## A selective gate traces a risk-coverage curve

Sweep the answer threshold over synthetic adequacy scores. Answering more buys coverage at rising selective risk; always-abstain attains trivially perfect risk with zero value.

In [ ]:
scores = [0.95, 0.90, 0.85, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20]
correct = [1, 1, 1, 1, 0, 1, 0, 0, 0, 0]


def operating_point(threshold):
    answered = [c for s, c in zip(scores, correct) if s >= threshold]
    coverage = len(answered) / len(scores)
    risk = (1 - sum(answered) / len(answered)) if answered else 0.0
    return round(coverage, 3), round(risk, 3)


curve = [(t, *operating_point(t)) for t in [0.0, 0.5, 0.8, 0.9, 1.01]]
print(f"{'threshold':>10s} {'coverage':>9s} {'sel_risk':>9s}")
for t, cov, risk in curve:
    print(f"{t:10.2f} {cov:9.3f} {risk:9.3f}")
print("always-abstain: coverage=0.000 risk=0.000 value=0.000")

In [ ]:
cov_all, risk_all = operating_point(0.0)
cov_none, risk_none = operating_point(1.01)
cov_mid, risk_mid = operating_point(0.8)
assert (cov_all, risk_all) == (1.0, 0.5)
assert cov_none == 0.0 and risk_none == 0.0
assert risk_mid <= risk_all and cov_mid > 0.0
assert operating_point(0.9)[1] <= operating_point(0.5)[1]
print(f"PASS: risk falls {risk_all:.2f} -> {risk_mid:.2f} as coverage {cov_all:.2f} -> {cov_mid:.2f}; abstain-all is safe and useless")

## Counterfactual answerability pair switches the route

Add the one decisive evidence item and the gate must flip from retrieve-with-gap to answer; remove it and it must flip back. Answering both is overconfidence, abstaining on both is overcaution.

In [ ]:
def gate(evidence_has_q3, user_gave_param=True, conflict=False):
    if conflict:
        return ("VERIFY", "conflicting sources")
    if not user_gave_param:
        return ("ASK", "user-owned parameter missing")
    if not evidence_has_q3:
        return ("RETRIEVE", "missing q3_revenue_value")
    return ("ANSWER", "all critical fields supported")


pair = {"without_q3": gate(False), "with_q3": gate(True)}
routes = {"underspecified": gate(True, user_gave_param=False), "conflict": gate(True, conflict=True)}
over_answerer = ("ANSWER", "ANSWER")
over_cautious = ("RETRIEVE", "RETRIEVE")
correct = (pair["without_q3"][0], pair["with_q3"][0])

print("answerability pair:", pair)
print("other routes:", routes)

In [ ]:
assert pair["without_q3"][0] == "RETRIEVE" and pair["with_q3"][0] == "ANSWER"
assert routes["underspecified"][0] == "ASK" and routes["conflict"][0] == "VERIFY"
assert over_answerer[0] == over_answerer[1] == "ANSWER"  # overconfident: never withholds
assert over_cautious[0] == over_cautious[1] == "RETRIEVE"  # overcautious: never answers
assert correct == ("RETRIEVE", "ANSWER")
print("PASS: the gate switches on the decisive item and routes by reason, not by phrase")

## What we earned

Epistemic adequacy is a typed, contract-relative property: coverage plus critical-field, conflict, freshness, and recoverability state. A selective gate converts it into a risk-coverage operating point, and counterfactual pairs verify the gate actually conditions on the decisive evidence.

Notebook 12 / Chapter 12 turns these diagnostic records into executable system behavior: from measurements to policy.